In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split

# Read the data
X_full = pd.read_csv('train.csv', index_col='Id')
X_test_full = pd.read_csv('test.csv', index_col='Id')

# Remove rows with missing target, separate target from predictors
X_full.dropna(axis=0, subset=['SalePrice'], inplace=True)
y = X_full.SalePrice
X_full.drop(['SalePrice'], axis=1, inplace=True)

# Break off validation set from training data
X_train_full, X_valid_full, y_train, y_valid = train_test_split(X_full, y, 
                                                                train_size=0.8, test_size=0.2,
                                                                random_state=0)

# "Cardinality" means the number of unique values in a column
# Select categorical columns with relatively low cardinality (convenient but arbitrary)
categorical_cols = [cname for cname in X_train_full.columns if
                    X_train_full[cname].nunique() < 10 and 
                    X_train_full[cname].dtype == "object"]

# Select numerical columns
numerical_cols = [cname for cname in X_train_full.columns if 
                X_train_full[cname].dtype in ['int64', 'float64']]

# Keep selected columns only
my_cols = categorical_cols + numerical_cols
X_train = X_train_full[my_cols].copy()
X_valid = X_valid_full[my_cols].copy()
X_test = X_test_full[my_cols].copy()

ModuleNotFoundError: No module named 'sklearn'

In [54]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error
from xgboost import XGBRegressor
from sklearn.model_selection import GridSearchCV
# Preprocessing for numerical data
numerical_transformer = SimpleImputer(strategy="mean") # Your code here

# Preprocessing for categorical data
categorical_transformer = Pipeline(steps=[("onehot",OneHotEncoder(handle_unknown='ignore')),("imputer",SimpleImputer(strategy="most_frequent"))]) # Your code here

# Bundle preprocessing for numerical and categorical data
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numerical_transformer, numerical_cols),
        ('cat', categorical_transformer, categorical_cols)
    ])

# Define model
model=XGBRegressor()

In [55]:
# Bundle preprocessing and modeling code in a pipeline
param_grid = [
    {
        "model__random_state":[0],
        "model__n_estimators": [238,239,240],
        "model__max_depth": [2,3,4],
        "model__learning_rate": [0.0791187,0.0791188,0.0791189],
        "model__grow_policy": ["depthwise"],
    }
]
#{'model__grow_policy': 'depthwise', 'model__learning_rate': 0.07, 'model__max_depth': 5, 'model__n_estimators': 200}
#17523.685546875

#{'model__grow_policy': 'depthwise', 'model__learning_rate': 0.07, 'model__max_depth': 4, 'model__n_estimators': 225}
#-16513.827799479168
#MAE: 17001.32421875

#{'model__grow_policy': 'depthwise', 'model__learning_rate': 0.08, 'model__max_depth': 3, 'model__n_estimators': 230}
#-16310.652994791666
#MAE: 17588.9140625

#{'model__grow_policy': 'depthwise', 'model__learning_rate': 0.08, 'model__max_depth': 3, 'model__n_estimators': 240}
#-16283.339518229166
#MAE: 17555.158203125

#{'model__grow_policy': 'depthwise', 'model__learning_rate': 0.08, 'model__max_depth': 3, 'model__n_estimators': 240, 'model__random_state': 0}
#-16283.339518229166
#MAE: 17555.158203125

#{'model__grow_policy': 'depthwise', 'model__learning_rate': 0.079, 'model__max_depth': 3, 'model__n_estimators': 240, 'model__random_state': 0}
#-16256.1064453125
#MAE: 17328.720703125

#{'model__grow_policy': 'depthwise', 'model__learning_rate': 0.0791, 'model__max_depth': 3, 'model__n_estimators': 240, 'model__random_state': 0}
#-16204.276611328125
#MAE: 16973.681640625

#{'model__grow_policy': 'depthwise', 'model__learning_rate': 0.07911, 'model__max_depth': 3, 'model__n_estimators': 240, 'model__random_state': 0}
#-16190.410888671875
#MAE: 17149.171875

#{'model__grow_policy': 'depthwise', 'model__learning_rate': 0.07912, 'model__max_depth': 3, 'model__n_estimators': 239, 'model__random_state': 0}
#-16129.884765625
#MAE: 16954.712890625

#{'model__grow_policy': 'depthwise', 'model__learning_rate': 0.079119, 'model__max_depth': 3, 'model__n_estimators': 239, 'model__random_state': 0}
#-16129.781494140625
#MAE: 16954.69140625

#{'model__grow_policy': 'depthwise', 'model__learning_rate': 0.0791188, 'model__max_depth': 3, 'model__n_estimators': 239, 'model__random_state': 0}
#-16129.779296875
#MAE: 16954.6875
my_pipeline = Pipeline(steps=[('preprocessor', preprocessor),
                              ('model', model)
                             ])
grid = GridSearchCV(
    estimator=my_pipeline,
    param_grid=param_grid,
    cv=4,
    scoring="neg_mean_absolute_error",
    n_jobs=-1
)
grid.fit(X_train, y_train)

print(grid.best_params_)
print(grid.best_score_)
best_model = grid.best_estimator_
preds=best_model.predict(X_valid)
score = mean_absolute_error(y_valid, preds)
print('MAE:', score)

{'model__grow_policy': 'depthwise', 'model__learning_rate': 0.0791187, 'model__max_depth': 3, 'model__n_estimators': 239, 'model__random_state': 0}
-16263.389892578125
MAE: 16988.36328125


In [51]:
# Preprocessing of test data, fit model
finalmodel=XGBRegressor(
    random_state=0,
    grow_policy='depthwise', 
    learning_rate=0.0791188, 
    max_depth=3, 
    n_estimators= 239
)
finalpipe = Pipeline(steps=[('preprocessor', preprocessor),
                              ('model', finalmodel)
                             ])
finalpipe.fit(X_full,y)
preds_test = finalpipe.predict(X_test_full) # Your code here

In [ ]:
# Save test predictions to file
output = pd.DataFrame({'Id': X_test.index,
                       'SalePrice': preds_test})
output.to_csv('submission.csv', index=False)